# YOLOv8n Training — Close-Proximity Drone Object Detection

Target deployment: **Raspberry Pi 3** + **Coral USB Accelerator (Edge TPU)**.

Because the final model has to run INT8-quantized on an Edge TPU on a weak host CPU, every choice below
(model size, input resolution, augmentation) is made backwards from that constraint, not from
"what gets the best mAP in Colab."

## Why this dataset needs to differ from generic aerial/UAV datasets
Public UAV datasets (VisDrone, etc.) are mostly high-altitude, top-down, tiny-object detection.
You're flying **close to objects**, so your training data needs:
- Objects occupying a **large fraction of the frame**, not tiny distant blobs
- **Partial occlusion / frame-edge truncation** (objects clip out of view as you approach)
- **Motion blur** (prop wash + fast maneuvers)
- Steep/varied **camera angles** relative to the object (not just nadir/top-down)
- Wide **lighting range** (outdoor flight = harsh sun, shadow, overcast)

If your labeled dataset doesn't already reflect this, augmentation can only partially compensate —
try to actually capture footage that mimics your real flight profile (close passes, various angles/speeds).


## 1. Environment setup

In [ ]:
!nvidia-smi


In [ ]:
!pip install -q ultralytics roboflow opencv-python-headless
import ultralytics
ultralytics.checks()


## 2. Turn raw footage into a labeling-ready frame set

You said you have raw footage but no labels yet. Don't label every frame of a video — consecutive
frames are ~duplicates and waste labeling effort. Extract frames at a fixed interval, biased toward
the close-approach portions of each clip where the object fills more of the frame.

Upload your raw video files to `/content/raw_videos/` before running this cell (or mount Drive).


In [ ]:
import cv2, os, glob

RAW_VIDEO_DIR = "/content/raw_videos"
FRAME_OUT_DIR = "/content/frames"
SAMPLE_EVERY_N_FRAMES = 6   # tune: lower = more frames extracted (more labeling work, more data)

os.makedirs(FRAME_OUT_DIR, exist_ok=True)

video_paths = glob.glob(os.path.join(RAW_VIDEO_DIR, "*"))
total_saved = 0

for vp in video_paths:
    cap = cv2.VideoCapture(vp)
    base = os.path.splitext(os.path.basename(vp))[0]
    frame_idx = 0
    saved_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_idx % SAMPLE_EVERY_N_FRAMES == 0:
            out_path = os.path.join(FRAME_OUT_DIR, f"{base}_{saved_idx:05d}.jpg")
            cv2.imwrite(out_path, frame)
            saved_idx += 1
            total_saved += 1
        frame_idx += 1
    cap.release()

print(f"Extracted {total_saved} frames from {len(video_paths)} video(s) into {FRAME_OUT_DIR}")


## 3. Label the frames (Roboflow)

Upload the contents of `/content/frames` to a new Roboflow project (Object Detection type) and label there —
it's the fastest path from "folder of images" to a YOLOv8-formatted dataset, and its auto-label / smart-polygon
tools speed up labeling raw footage a lot.

**Augmentation settings to enable in Roboflow, given your close-proximity use case:**
- Blur (mimics motion blur from prop wash / fast movement)
- Brightness / Exposure jitter (outdoor lighting variance)
- Cutout (simulates partial occlusion as objects exit frame near the drone)
- Rotation ± 15–20° (camera angle variety during approach)
- Skip aggressive zoom-out augmentation — your real deployment rarely sees tiny distant objects, so
  don't let the augmented dataset over-represent that case at the expense of close-range examples.

Once labeled and versioned, generate the export and drop your snippet into the next cell.


In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1)  # set to your dataset version number
dataset = version.download("yolov8")

print(dataset.location)


## 4. Train

Key choices made for the Pi 3 + Coral USB Accelerator target, not for max accuracy:

- **`yolov8n.pt`** — the nano model. Anything larger doesn't make sense here: it'll dominate the
  INT8 export size/latency and a Pi 3's CPU is already the bottleneck for pre/post-processing regardless
  of accelerator, so keep the model itself as light as possible.
- **`imgsz=320`** — smaller than the notebook default (640). Edge TPU throughput and Pi 3's frame
  pre-processing cost both scale with resolution; 320 is a good balance of detection quality vs. speed
  for close-range objects (which are large in-frame anyway, so you don't need 640's fine detail).
- **`close_mosaic=10`** — disables mosaic augmentation for the last 10 epochs so the model trains on
  more realistic (unmosaicked) framing right before it converges.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=320,
    batch=16,
    patience=20,
    close_mosaic=10,
    device=0,
    project="uav_close_proximity",
    name="yolov8n_320",
)


## 5. Validate

In [ ]:
metrics = model.val()
print(metrics.box.map)     # mAP50-95
print(metrics.box.map50)   # mAP50


## 6. Export to INT8 TFLite (Edge TPU compatible)

`int8=True` runs post-training quantization using your training images as the representative dataset.
This is the format the Edge TPU compiler needs — full-precision or float16 models will not run on the
Coral accelerator.


In [ ]:
model.export(format="tflite", int8=True, imgsz=320, data=f"{dataset.location}/data.yaml")


## 7. Compile for Edge TPU

This step turns the standard INT8 TFLite model into one that actually offloads to the Coral accelerator
(operations it can't map to the TPU fall back to the Pi's CPU — the compiler's log tells you if that happens,
worth checking since CPU fallback ops are exactly the throughput killer you're trying to avoid).


In [ ]:
!curl -s https://packages.cloud.google.com/apt/doc/apt-key.gpg | apt-key add -
!echo "deb https://packages.cloud.google.com/apt coral-edgetpu-stable main" | tee /etc/apt/sources.list.d/coral-edgetpu.list
!apt-get update -qq && apt-get install -qq edgetpu-compiler

import glob
tflite_path = glob.glob("uav_close_proximity/yolov8n_320/weights/best_saved_model/*_int8.tflite")[0]
!edgetpu_compiler -s {tflite_path}


## 8. Download

Grab the `*_edgetpu.tflite` file this produces — that's what goes on the Pi 3 with the Coral inference
script. Also keep the plain (non-edgetpu) `best.pt` around in case you want to re-export later for a
different target.


In [ ]:
from google.colab import files
import glob

edgetpu_model = glob.glob("*_edgetpu.tflite")[0]
files.download(edgetpu_model)
